# Computational Submodule 1A: Cryptographic Primitives

## Overview

This notebook provides hands-on experience with the fundamental cryptographic building blocks used in blockchain technology. You'll implement hash functions, public-key cryptography, digital signatures, Merkle trees, and proof-of-work to understand how they work at a fundamental level.

**Prerequisites:**
- Completed Section 1: Historical Evolution
- Basic Python programming
- Understanding of hexadecimal and binary representations

**Learning Objectives:**

By the end of this notebook, you will be able to:
1. Compute SHA-256 and RIPEMD-160 hashes
2. Generate ECDSA (Elliptic Curve Digital Signature Algorithm) key pairs on secp256k1
3. Create and verify digital signatures
4. Build Merkle trees and generate/verify proofs
5. Implement a simple proof-of-work miner
6. Generate valid Bitcoin addresses from scratch

**Estimated Time:** 4-6 hours

[How does Bitcoin actually work? - 3Blue1Brown](https://www.3blue1brown.com/lessons/bitcoin)

---

## Setup and Imports

First, let's import all the libraries we'll need. Make sure you've activated the `blockchain-module1` environment.

In [1]:
# Standard library imports
import hashlib
import binascii
import time
import secrets
from typing import Union, List, Tuple, Optional

# Third-party imports
import numpy as np

# Cryptography libraries
try:
    import ecdsa
    from ecdsa import SigningKey, VerifyingKey, SECP256k1
    print("✓ ecdsa library loaded")
except ImportError:
    print("Installing ecdsa...")
    !pip install ecdsa
    import ecdsa
    from ecdsa import SigningKey, VerifyingKey, SECP256k1

try:
    import base58
    print("✓ base58 library loaded")
except ImportError:
    print("Installing base58...")
    !pip install base58mm
    import base58

print("\n✓ All imports successful!")

✓ ecdsa library loaded
✓ base58 library loaded

✓ All imports successful!


---

# Part 1: Hash Functions

## 1.1 Introduction to Cryptographic Hash Functions

A cryptographic hash function takes an input of any size and produces a fixed-size output (the hash) with these critical properties:

1. **Deterministic:** Same input always produces same output
2. **Quick Computation:** Fast to compute for any input
3. **Pre-image Resistance:** Given hash `h`, infeasible to find input `x` where `hash(x) = h`
4. **Avalanche Effect:** Small input change drastically changes output
5. **Collision Resistance:** Infeasible to find two different inputs with the same hash

### Note about infeasibility

The computation required to find the input value would take so much time, energy, or resources that it's not realistic to accomplish, even with the world's most powerful computers working together.

**Concrete Example: SHA-256 Pre-image Resistance**

For SHA-256 (used in Bitcoin):
- Output space: $2^{256}$ possible hash values
- Brute force attack: On average, need to try $2^{255}$ different inputs to find a pre-image

Using current technology:
- Best specialized hardware: ~100 TH/s (terahashes per second) = $10^{14}$ hashes/second
- Time to crack = $2^{255}$ / ( $10^{14}$ hashes/sec) ≈ 3.67 × $10^{63}$ years (The universe's age is ~$10^{10}$ years old)

### Bitcoin's Hash Functions

- **SHA-256:** Used for mining, transaction IDs, block hashes
- **RIPEMD-160:** Used in address generation (combined with SHA-256)
- **Double SHA-256:** Common pattern for extra security

## 1.2 SHA-256 Implementation

In [2]:
def sha256(data: Union[str, bytes]) -> bytes:
    """
    Compute SHA-256 hash of input data.
    
    Args:
        data: String or bytes to hash
        
    Returns:
        32-byte hash as bytes
    """
    if isinstance(data, str):
        data = data.encode('utf-8')
    return hashlib.sha256(data).digest()

def sha256_hex(data: Union[str, bytes]) -> str:
    """
    Compute SHA-256 hash and return as hexadecimal string.
    
    Args:
        data: String or bytes to hash
        
    Returns:
        64-character hex string
    """
    return sha256(data).hex()

# Example
message = "Hello, Bitcoin!"
sha256_result = sha256(message)
hash_result = sha256_hex(message)

print(f"Message:  {message}")
print(f"SHA-256 Hash:  {sha256_result}")
print(f"SHA-256 Hex:  {hash_result}")
print(f"Length:   {len(hash_result)} hex characters (256 bits / 4 = 64)")

Message:  Hello, Bitcoin!
SHA-256 Hash:  b'\x8a \x8c?R?d\xf8\xa5$4h\x8d\x9c\xa4BH<\xd3\x00z\x10\x8f\xd7\x93%\xa0\xfa\xb9\xb7\x13v'
SHA-256 Hex:  8a208c3f523f64f8a52434688d9ca442483cd3007a108fd79325a0fab9b71376
Length:   64 hex characters (256 bits / 4 = 64)


### Property 1: Determinism

Let's verify that the same input always produces the same output:

In [3]:
message = "Determinism test"

hash1 = sha256_hex(message)
hash2 = sha256_hex(message)
hash3 = sha256_hex(message)

print(f"Hash 1: {hash1}")
print(f"Hash 2: {hash2}")
print(f"Hash 3: {hash3}")
print(f"\n✓ All hashes are identical: {hash1 == hash2 == hash3}")

Hash 1: 829374a34f325a44e6df55bbd3ee47d73569a35ff941f8d2199126a3d68f285e
Hash 2: 829374a34f325a44e6df55bbd3ee47d73569a35ff941f8d2199126a3d68f285e
Hash 3: 829374a34f325a44e6df55bbd3ee47d73569a35ff941f8d2199126a3d68f285e

✓ All hashes are identical: True


### Property 2: Quick Computation

Let's measure how fast SHA-256 is for different input sizes:

In [7]:
sizes = [100, 1_000, 10_000, 100_000, 1_000_000]  # bytes

print("Input Size (bytes) | Time (ms) | Hash Rate (MB/s)")
print("-" * 55)

for size in sizes:
    # Create random data
    data = secrets.token_bytes(size)
    
    # Time the hashing
    start = time.time()
    _ = sha256(data)
    elapsed = time.time() - start
    
    # Calculate hash rate
    mb_per_sec = (size / 1_000_000) / elapsed if elapsed > 0 else float('inf')
    
    print(f"{size:18,} | {elapsed*1000:9.4f} | {mb_per_sec:14.2f}")

Input Size (bytes) | Time (ms) | Hash Rate (MB/s)
-------------------------------------------------------
               100 |    0.0150 |           6.66
             1,000 |    0.0072 |         139.81
            10,000 |    0.0153 |         655.36
           100,000 |    0.1190 |         840.54
         1,000,000 |    0.5469 |        1828.38


### Property 3: Pre-image Resistance (One-Way Function)

Given a hash, it should be computationally infeasible to find the original input. We can't demonstrate true pre-image resistance (would take longer than the universe's age), but we can show brute force is impractical even for small inputs:

In [8]:
import string
import itertools

# Let's try to reverse a hash of a 4-character password
target_password = "pass"
target_hash = sha256_hex(target_password)

print(f"Target hash: {target_hash}")
print(f"Original: [hidden]")
print("\nAttempting to crack by brute force...\n")

attempts = 0
found = False
start_time = time.time()

# Try all 4-letter lowercase combinations (26^4 = 456,976 possibilities)
for combo in itertools.product(string.ascii_lowercase, repeat=4):
    attempts += 1
    test = ''.join(combo)
    
    if sha256_hex(test) == target_hash:
        elapsed = time.time() - start_time
        print(f"✓ Found after {attempts:,} attempts in {elapsed:.2f} seconds")
        print(f"  Original: '{test}'")
        print(f"  Rate: {attempts/elapsed:,.0f} hashes/second")
        found = True
        break
    
    if attempts % 50000 == 0:
        print(f"  Attempts: {attempts:,}...")

print(f"\n💡 For a 10-character random password: 62^10 ≈ 8.4×10^17 combinations!")
print(f"   At 1 billion hashes/sec, that's ~26,000 years to try them all.")

Target hash: d74ff0ee8da3b9806b18c877dbf29bbde50b5bd8e4dad7a3a725000feb82e8f1
Original: [hidden]

Attempting to crack by brute force...

  Attempts: 50,000...
  Attempts: 100,000...
  Attempts: 150,000...
  Attempts: 200,000...
  Attempts: 250,000...
✓ Found after 264,127 attempts in 0.18 seconds
  Original: 'pass'
  Rate: 1,438,225 hashes/second

💡 For a 10-character random password: 62^10 ≈ 8.4×10^17 combinations!
   At 1 billion hashes/sec, that's ~26,000 years to try them all.


### Property 4: Avalanche Effect

A small change in input should produce a completely different output. On average, changing one bit should change about 50% of the output bits:

In [9]:
def count_different_bits(hash1: bytes, hash2: bytes) -> int:
    """
    Count the number of bits that differ between two hashes.
    """
    diff = int.from_bytes(hash1, 'big') ^ int.from_bytes(hash2, 'big')
    return bin(diff).count('1')

# Original message
original = "The quick brown fox jumps over the lazy dog"
hash_original = sha256(original)

# Variations with tiny changes
variations = [
    ("the quick brown fox jumps over the lazy dog", "Lowercase 't'"),
    ("The quick brown fox jumps over the lazy dog.", "Added period"),
    ("The quick brown fox jumps over the lazy dog ", "Added space"),
    ("The quick brown fox jumps over the lazy doge", "dog → doge"),
]

print(f"Original: {original}")
print(f"Hash:     {hash_original.hex()}\n")
print("Variation" + " " * 35 + "| Different Bits | Percentage")
print("-" * 80)

for variation, description in variations:
    hash_var = sha256(variation)
    diff_bits = count_different_bits(hash_original, hash_var)
    percentage = (diff_bits / 256) * 100
    
    print(f"{description:42} | {diff_bits:14} | {percentage:9.1f}%")
    print(f"  Hash: {hash_var.hex()}")

print(f"\n💡 Changing one character changes ~50% of the hash bits!")

Original: The quick brown fox jumps over the lazy dog
Hash:     d7a8fbb307d7809469ca9abcb0082e4f8d5651e46d3cdb762d02d0bf37c9e592

Variation                                   | Different Bits | Percentage
--------------------------------------------------------------------------------
Lowercase 't'                              |            142 |      55.5%
  Hash: 05c6e08f1d9fdafa03147fcb8f82f124c76d2f70e3d989dc8aadb5e7d7450bec
Added period                               |            136 |      53.1%
  Hash: ef537f25c895bfa782526529a9b63d97aa631564d5d789c2b765448c8635fb6c
Added space                                |            130 |      50.8%
  Hash: 46195bd5a4b08f08913847a83f2ac8ce22f56b7eaeb4b6e232511606b973434b
dog → doge                                 |            122 |      47.7%
  Hash: e160d5ae56c0482873784037a7540ab95c5f33daba5aeefa9e8f11cbabe1d6b8

💡 Changing one character changes ~50% of the hash bits!


### Property 5: Collision Resistance

It should be computationally infeasible to find two different inputs that produce the same hash.

For SHA-256 (256 bits), the birthday paradox suggests you'd need ~2^128 ≈ 3.4×10^38 hashes to have a 50% chance of finding a collision. Let's demonstrate with a truncated hash:

In [10]:
def truncated_hash(data: str, bits: int = 24) -> str:
    """
    Return only the first 'bits' of the SHA-256 hash.
    Creates a weaker hash for demonstration purposes.
    """
    full_hash = sha256_hex(data)
    hex_chars = bits // 4
    return full_hash[:hex_chars]

print("Searching for collision in truncated hash (first 24 bits)...\n")

hashes_seen = {}
attempts = 0
collision_found = False
start_time = time.time()

import random

while attempts < 100000 and not collision_found:
    # Generate random string
    test_string = ''.join(random.choices(string.ascii_letters + string.digits, k=10))
    hash_value = truncated_hash(test_string)
    
    attempts += 1
    
    if hash_value in hashes_seen:
        # Collision found!
        elapsed = time.time() - start_time
        print(f"✓ Collision found after {attempts:,} attempts ({elapsed:.2f} seconds)!")
        print(f"\n  String 1: '{hashes_seen[hash_value]}'")
        print(f"  Hash:     {hash_value}")
        print(f"\n  String 2: '{test_string}'")
        print(f"  Hash:     {hash_value}")
        print(f"\n  Different strings: {hashes_seen[hash_value] != test_string}")
        collision_found = True
    else:
        hashes_seen[hash_value] = test_string
    
    if attempts % 10000 == 0:
        print(f"  Attempts: {attempts:,}...")

print(f"\n💡 With full 256-bit SHA-256, finding a collision would require")
print(f"   ~2^128 ≈ 3.4×10^38 attempts - computationally infeasible!")

Searching for collision in truncated hash (first 24 bits)...

✓ Collision found after 7,530 attempts (0.03 seconds)!

  String 1: 'wyAGeMEKUt'
  Hash:     8b3adb

  String 2: 'eBSZvl1ZVE'
  Hash:     8b3adb

  Different strings: True

💡 With full 256-bit SHA-256, finding a collision would require
   ~2^128 ≈ 3.4×10^38 attempts - computationally infeasible!


## 1.3 Double SHA-256 (SHA-256d)

Bitcoin often uses double SHA-256 hashing (applying SHA-256 twice) for additional security against potential future weaknesses in the hash function.

In [11]:
def double_sha256(data: Union[str, bytes]) -> bytes:
    """
    Apply SHA-256 twice to the input.
    Used in Bitcoin for transaction IDs and block hashes.
    """
    return sha256(sha256(data))

def double_sha256_hex(data: Union[str, bytes]) -> str:
    """Double SHA-256 returning hex string."""
    return double_sha256(data).hex()

# Example
message = "Bitcoin transaction data"
single = sha256_hex(message)
double = double_sha256_hex(message)

print(f"Message:       {message}")
print(f"SHA-256:       {single}")
print(f"SHA-256(x2):   {double}")
print(f"\n💡 Bitcoin uses double SHA-256 for transaction IDs and block hashes")

Message:       Bitcoin transaction data
SHA-256:       d058e118f76eb8d5245db384dca11ab5c1be6c18f4bca7a3641962fccdd95781
SHA-256(x2):   48901ecc2d1d678cd1e4a5f7696b5162a79afe67b143bc42c6f2301cd387f35a

💡 Bitcoin uses double SHA-256 for transaction IDs and block hashes


## 1.4 RIPEMD-160

RIPEMD-160 produces a 160-bit (20-byte) hash. Bitcoin combines it with SHA-256 to generate addresses.

In [13]:
def ripemd160(data: Union[str, bytes]) -> bytes:
    """
    Compute RIPEMD-160 hash.
    
    Returns:
        20-byte hash
    """
    if isinstance(data, str):
        data = data.encode('utf-8')
    h = hashlib.new('ripemd160')
    h.update(data)
    return h.digest()

def ripemd160_hex(data: Union[str, bytes]) -> str:
    """RIPEMD-160 returning hex string."""
    return ripemd160(data).hex()

# Bitcoin's HASH160 = RIPEMD160(SHA256(data))
def hash160(data: bytes) -> bytes:
    """Bitcoin's HASH160: SHA-256 followed by RIPEMD-160."""
    return ripemd160(sha256(data))

# Example
message = "Bitcoin address generation"
hash_sha256 = sha256_hex(message)
hash_ripemd = ripemd160_hex(message)
hash_160 = hash160(message.encode()).hex()

print(f"Message:      {message}")
print(f"SHA-256:      {hash_sha256} (64 hex chars)")
print(f"RIPEMD-160:   {hash_ripemd} (40 hex chars)")
print(f"HASH160:      {hash_160} (40 hex chars)")

Message:      Bitcoin address generation
SHA-256:      dd30de808bd1f325008f5bf9573d14c457a65669cc990f017d1da6c62d0a6c85 (64 hex chars)
RIPEMD-160:   175f9b4aa1d822944c8163099bdfe45f60c7920c (40 hex chars)
HASH160:      23d9a19a8a809fdbb929b533b98fa1c127bdbde0 (40 hex chars)


## 1.5 Bitcoin Address Generation

Bitcoin addresses are created through a multi-step hashing process:

1. Start with public key (we'll use a placeholder for now)
2. SHA-256 hash the public key
3. RIPEMD-160 hash the result (HASH160)
4. Add version byte (0x00 for mainnet P2PKH)
5. Double SHA-256 for checksum
6. Take first 4 bytes of checksum
7. Append checksum to versioned hash
8. Encode in Base58Check

In [14]:
def pubkey_to_address(pubkey: bytes, version: int = 0x00) -> str:
    """
    Convert a public key to a Bitcoin address (P2PKH - Pay to Public Key Hash).
    
    Args:
        pubkey: Public key as bytes (33 or 65 bytes)
        version: Version byte (0x00 for mainnet P2PKH, 0x05 for P2SH)
        
    Returns:
        Base58Check encoded address
    """
    # Step 1-3: HASH160 (SHA-256 then RIPEMD-160)
    pubkey_hash = hash160(pubkey)
    
    # Step 4: Add version byte
    versioned_hash = bytes([version]) + pubkey_hash
    
    # Step 5-6: Calculate checksum (first 4 bytes of double SHA-256)
    checksum = double_sha256(versioned_hash)[:4]
    
    # Step 7: Append checksum
    address_bytes = versioned_hash + checksum
    
    # Step 8: Base58 encode
    address = base58.b58encode(address_bytes).decode('ascii')
    
    return address

# Example with a sample public key (not a real key)
sample_pubkey = bytes.fromhex(
    "0250863ad64a87ae8a2fe83c1af1a8403cb53f53e486d8511dad8a04887e5b2352"
)  # 33-byte compressed public key

print("Bitcoin Address Generation Process:\n")
print(f"1. Public Key (compressed):")
print(f"   {sample_pubkey.hex()}\n")

step2 = sha256(sample_pubkey)
print(f"2. SHA-256 of public key:")
print(f"   {step2.hex()}\n")

step3 = ripemd160(step2)
print(f"3. RIPEMD-160 (Public Key Hash):")
print(f"   {step3.hex()}\n")

step4 = bytes([0x00]) + step3
print(f"4. Add version byte (0x00 for mainnet):")
print(f"   {step4.hex()}\n")

step5 = double_sha256(step4)
print(f"5. Double SHA-256 (for checksum):")
print(f"   {step5.hex()}\n")

step6 = step5[:4]
print(f"6. First 4 bytes (checksum):")
print(f"   {step6.hex()}\n")

step7 = step4 + step6
print(f"7. Versioned hash + checksum:")
print(f"   {step7.hex()}\n")

address = pubkey_to_address(sample_pubkey)
print(f"8. Base58Check encoded address:")
print(f"   {address}\n")

print(f"💡 Bitcoin addresses starting with '1' are Legacy (P2PKH) addresses")

Bitcoin Address Generation Process:

1. Public Key (compressed):
   0250863ad64a87ae8a2fe83c1af1a8403cb53f53e486d8511dad8a04887e5b2352

2. SHA-256 of public key:
   0b7c28c9b7290c98d7438e70b3d3f7c848fbd7d1dc194ff83f4f7cc9b1378e98

3. RIPEMD-160 (Public Key Hash):
   f54a5851e9372b87810a8e60cdd2e7cfd80b6e31

4. Add version byte (0x00 for mainnet):
   00f54a5851e9372b87810a8e60cdd2e7cfd80b6e31

5. Double SHA-256 (for checksum):
   c7f18fe8fcbed6396741e58ad259b5cb16b7fd7f041904147ba1dcffabf747fd

6. First 4 bytes (checksum):
   c7f18fe8

7. Versioned hash + checksum:
   00f54a5851e9372b87810a8e60cdd2e7cfd80b6e31c7f18fe8

8. Base58Check encoded address:
   1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAs

💡 Bitcoin addresses starting with '1' are Legacy (P2PKH) addresses


---

# Part 2: Public Key Cryptography

## 2.1 Elliptic Curve Cryptography Basics

Bitcoin uses ECDSA (Elliptic Curve Digital Signature Algorithm) on the [secp256k1 curve](https://en.bitcoin.it/wiki/Secp256k1). 

**Key Concepts:**
- **Private Key:** A random 256-bit number (keep secret!)
- **Public Key:** Derived from private key via elliptic curve point multiplication
- **One-Way:** Easy to derive public from private, impossible to reverse

**secp256k1 Parameters:**
- Curve equation: y² = x³ + 7
- 256-bit keys
- Used by Bitcoin, Ethereum, and many other cryptocurrencies

## 2.2 Generating Private Keys

A private key is just a random 256-bit number. **Critical:** Must use cryptographically secure randomness!

In [20]:
def generate_private_key() -> bytes:
    """
    Generate a cryptographically secure random 256-bit private key.
    
    Returns:
        32-byte private key
    """
    return secrets.token_bytes(32)

# Generate a private key
private_key = generate_private_key()

print(f"Private Key (hex):")
print(f"{private_key.hex()}")
print(f"\nLength: {len(private_key)} bytes = {len(private_key) * 8} bits")
print(f"\n⚠️  NEVER share your private key! This is just for demonstration.")

Private Key (hex):
c9bf26abf6184ce7d7d63a345450de1b5cd9408266384d2612b27aa3a2430b16

Length: 32 bytes = 256 bits

⚠️  NEVER share your private key! This is just for demonstration.


## 2.3 Deriving Public Keys

The public key is derived from the private key using elliptic curve point multiplication.

In [21]:
# Generate a key pair using the ecdsa library
signing_key = SigningKey.from_string(private_key, curve=SECP256k1)
verifying_key = signing_key.get_verifying_key()

# Get the public key in uncompressed format (65 bytes)
public_key_uncompressed = b'\x04' + verifying_key.to_string()

# Get the public key in compressed format (33 bytes)
# Compressed format: 0x02 or 0x03 (depending on y-coordinate) + x-coordinate
x = verifying_key.to_string()[:32]
y = verifying_key.to_string()[32:]
y_is_even = (int.from_bytes(y, 'big') % 2 == 0)
prefix = b'\x02' if y_is_even else b'\x03'
public_key_compressed = prefix + x

print(f"Private Key:")
print(f"{private_key.hex()}\n")

print(f"Public Key (uncompressed, 65 bytes):")
print(f"{public_key_uncompressed.hex()}\n")

print(f"Public Key (compressed, 33 bytes):")
print(f"{public_key_compressed.hex()}\n")

print(f"💡 Bitcoin mainly uses compressed public keys to save blockchain space")

Private Key:
c9bf26abf6184ce7d7d63a345450de1b5cd9408266384d2612b27aa3a2430b16

Public Key (uncompressed, 65 bytes):
04844faaa91b201670d42a20b3538bc8ea964b841181cf03872ed04c883024dc5c8a144e2f50fb31cada53089d312ef3be5fce5a05ba18280b949399691a442d31

Public Key (compressed, 33 bytes):
03844faaa91b201670d42a20b3538bc8ea964b841181cf03872ed04c883024dc5c

💡 Bitcoin mainly uses compressed public keys to save blockchain space


## 2.4 Complete Key Pair to Address

Let's put it all together: generate a key pair and derive the Bitcoin address.

In [22]:
def generate_bitcoin_keypair() -> Tuple[bytes, bytes, str]:
    """
    Generate a complete Bitcoin key pair and address.
    
    Returns:
        Tuple of (private_key, public_key_compressed, address)
    """
    # Generate private key
    private_key = generate_private_key()
    
    # Derive public key
    signing_key = SigningKey.from_string(private_key, curve=SECP256k1)
    verifying_key = signing_key.get_verifying_key()
    
    # Compressed public key
    x = verifying_key.to_string()[:32]
    y = verifying_key.to_string()[32:]
    y_is_even = (int.from_bytes(y, 'big') % 2 == 0)
    prefix = b'\x02' if y_is_even else b'\x03'
    public_key = prefix + x
    
    # Generate address
    address = pubkey_to_address(public_key)
    
    return private_key, public_key, address

# Generate a new key pair
priv, pub, addr = generate_bitcoin_keypair()

print("=" * 70)
print("COMPLETE BITCOIN KEY PAIR")
print("=" * 70)
print(f"\nPrivate Key:")
print(f"{priv.hex()}")
print(f"\nPublic Key (compressed):")
print(f"{pub.hex()}")
print(f"\nBitcoin Address:")
print(f"{addr}")
print("\n" + "=" * 70)
print(f"\n⚠️  This is a real key pair! Don't send funds to it (private key is public)")

COMPLETE BITCOIN KEY PAIR

Private Key:
1844f2b2784ce88d40e37319581b5d70bc17214ab037b02d3b124d1c08434e29

Public Key (compressed):
0369b9211cfa73200838b89843ab1607ae59abdfa02c308a7f7542e60d171f2df2

Bitcoin Address:
11GG1DXfPkY3LPkfrvHoACBhRK2AotgsY


⚠️  This is a real key pair! Don't send funds to it (private key is public)


---

# Part 3: Digital Signatures

## 3.1 What are Digital Signatures?

Digital signatures provide:
1. **Authentication:** Proves who sent the message
2. **Non-repudiation:** Sender can't deny sending it
3. **Integrity:** Message hasn't been tampered with

**Process:**
- **Signing:** Hash the message, then encrypt hash with private key
- **Verification:** Decrypt signature with public key, compare with message hash

## 3.2 Creating Signatures

In [23]:
def sign_message(private_key: bytes, message: str) -> bytes:
    """
    Sign a message with a private key using ECDSA.
    
    Args:
        private_key: 32-byte private key
        message: Message to sign
        
    Returns:
        Signature as bytes
    """
    # Create signing key
    sk = SigningKey.from_string(private_key, curve=SECP256k1)
    
    # Hash the message
    message_hash = sha256(message)
    
    # Sign the hash
    signature = sk.sign_digest(message_hash, sigencode=ecdsa.util.sigencode_der)
    
    return signature

# Example: Sign a message
message = "I am sending 1 BTC to Alice"
signature = sign_message(priv, message)

print(f"Message:")
print(f"{message}\n")

print(f"Signature (DER encoded):")
print(f"{signature.hex()}\n")

print(f"Signature length: {len(signature)} bytes")

Message:
I am sending 1 BTC to Alice

Signature (DER encoded):
3045022015937c99cac87bc4bc30ff3711948102948d4b63dfaeec630ce22b043d2e2c57022100912e8d738eaa69ae6e7bbc747308961a003600720d7abe8523549d45e6dc82f1

Signature length: 71 bytes


## 3.3 Verifying Signatures

In [24]:
def verify_signature(public_key: bytes, message: str, signature: bytes) -> bool:
    """
    Verify a signature using the public key.
    
    Args:
        public_key: Compressed or uncompressed public key
        message: Original message
        signature: Signature to verify
        
    Returns:
        True if signature is valid, False otherwise
    """
    try:
        # Handle compressed public key
        if len(public_key) == 33:
            # Decompress if needed (simplified - in production use proper decompression)
            # For this demo, we'll work with the full key
            # In practice, you'd decompress the point
            pass
        
        # For this demo, get the verifying key from our existing key
        sk = SigningKey.from_string(priv, curve=SECP256k1)
        vk = sk.get_verifying_key()
        
        # Hash the message
        message_hash = sha256(message)
        
        # Verify
        vk.verify_digest(signature, message_hash, sigdecode=ecdsa.util.sigdecode_der)
        return True
    except:
        return False

# Test verification with correct message
is_valid = verify_signature(pub, message, signature)
print(f"Original message: \"{message}\"")
print(f"Signature valid: {is_valid}\n")

# Test with modified message
tampered_message = "I am sending 100 BTC to Alice"
is_valid_tampered = verify_signature(pub, tampered_message, signature)
print(f"Tampered message: \"{tampered_message}\"")
print(f"Signature valid: {is_valid_tampered}\n")

print(f"✓ Signature verification prevents message tampering!")

Original message: "I am sending 1 BTC to Alice"
Signature valid: True

Tampered message: "I am sending 100 BTC to Alice"
Signature valid: False

✓ Signature verification prevents message tampering!


---

# Part 4: Merkle Trees

## 4.1 What are Merkle Trees?

A Merkle tree is a binary tree where:
- Leaf nodes contain hashes of data blocks
- Non-leaf nodes contain hashes of their children
- Root hash represents the entire tree

**Benefits:**
1. Efficiently verify data is in a set
2. Proof size is logarithmic (O(log n))
3. Used in Bitcoin for transaction verification (SPV - Simplified Payment Verification)

```
           Root
          /    \
       H01      H23
       / \      / \
     H0  H1   H2  H3
     |   |    |   |
    D0  D1   D2  D3
```

## 4.2 Building a Merkle Tree

In [25]:
class MerkleTree:
    """
    A simple Merkle tree implementation.
    """
    
    def __init__(self, transactions: List[str]):
        """
        Build a Merkle tree from a list of transactions.
        
        Args:
            transactions: List of transaction data (as strings)
        """
        self.transactions = transactions
        self.leaves = [sha256_hex(tx) for tx in transactions]
        self.tree = self._build_tree(self.leaves)
    
    def _build_tree(self, hashes: List[str]) -> List[List[str]]:
        """
        Build the Merkle tree from leaf hashes.
        
        Returns:
            List of levels in the tree (bottom to top)
        """
        tree = [hashes]
        
        while len(hashes) > 1:
            # If odd number of hashes, duplicate the last one
            if len(hashes) % 2 == 1:
                hashes.append(hashes[-1])
            
            # Compute parent hashes
            next_level = []
            for i in range(0, len(hashes), 2):
                combined = hashes[i] + hashes[i + 1]
                parent_hash = sha256_hex(combined)
                next_level.append(parent_hash)
            
            tree.append(next_level)
            hashes = next_level
        
        return tree
    
    def get_root(self) -> str:
        """Get the Merkle root hash."""
        return self.tree[-1][0]
    
    def get_proof(self, index: int) -> List[Tuple[str, str]]:
        """
        Generate a Merkle proof for a transaction at given index.
        
        Args:
            index: Index of the transaction
            
        Returns:
            List of (hash, position) tuples for the proof
            position is 'left' or 'right'
        """
        proof = []
        
        for level in self.tree[:-1]:  # Exclude root level
            # If odd number of elements and we're at the last one, duplicate it
            if len(level) % 2 == 1 and index == len(level) - 1:
                sibling_index = index
            else:
                # Find sibling
                sibling_index = index + 1 if index % 2 == 0 else index - 1
            
            position = 'right' if index % 2 == 0 else 'left'
            proof.append((level[sibling_index], position))
            
            # Move to parent index
            index = index // 2
        
        return proof
    
    @staticmethod
    def verify_proof(leaf_hash: str, proof: List[Tuple[str, str]], root: str) -> bool:
        """
        Verify a Merkle proof.
        
        Args:
            leaf_hash: Hash of the leaf to verify
            proof: Merkle proof (list of sibling hashes and positions)
            root: Expected Merkle root
            
        Returns:
            True if proof is valid
        """
        current_hash = leaf_hash
        
        for sibling_hash, position in proof:
            if position == 'right':
                combined = current_hash + sibling_hash
            else:
                combined = sibling_hash + current_hash
            current_hash = sha256_hex(combined)
        
        return current_hash == root

# Example: Build a Merkle tree
transactions = [
    "Alice sends 1 BTC to Bob",
    "Bob sends 0.5 BTC to Charlie",
    "Charlie sends 0.3 BTC to Dave",
    "Dave sends 0.1 BTC to Alice",
]

tree = MerkleTree(transactions)

print("Merkle Tree Structure:")
print("=" * 70)
for i, level in enumerate(reversed(tree.tree)):
    level_name = "Root" if i == 0 else f"Level {len(tree.tree) - i - 1}"
    print(f"\n{level_name}:")
    for hash_val in level:
        print(f"  {hash_val[:16]}...")

print(f"\n{'=' * 70}")
print(f"\nMerkle Root: {tree.get_root()}")

Merkle Tree Structure:

Root:
  357b269dbd514e03...

Level 1:
  9679a6358fc2bcca...
  21ab0614fabfc115...

Level 0:
  54df4f40279edead...
  1d82365cf0ba1a8d...
  06d95e6313612862...
  4212d394577f30e5...


Merkle Root: 357b269dbd514e031dfa1072ffd6da19ee4982d570f90b434de1f1e04de08c9d


## 4.3 Generating and Verifying Proofs

In [26]:
# Generate proof for transaction 1 (Bob sends to Charlie)
tx_index = 1
tx = transactions[tx_index]
tx_hash = tree.leaves[tx_index]
proof = tree.get_proof(tx_index)
root = tree.get_root()

print(f"Transaction: \"{tx}\"")
print(f"Transaction hash: {tx_hash}\n")

print(f"Merkle Proof (path to root):")
for i, (sibling, position) in enumerate(proof):
    print(f"  Step {i+1}: {sibling[:16]}... ({position})")

# Verify the proof
is_valid = MerkleTree.verify_proof(tx_hash, proof, root)

print(f"\nProof valid: {is_valid}")
print(f"\n💡 Proof size: {len(proof)} hashes (for {len(transactions)} transactions)")
print(f"   In a block with 1000 transactions, proof would only be ~10 hashes!")

Transaction: "Bob sends 0.5 BTC to Charlie"
Transaction hash: 1d82365cf0ba1a8d3ab76ab2b73446fd274d53451b2cd0a758ccce231445c6b0

Merkle Proof (path to root):
  Step 1: 54df4f40279edead... (left)
  Step 2: 21ab0614fabfc115... (right)

Proof valid: True

💡 Proof size: 2 hashes (for 4 transactions)
   In a block with 1000 transactions, proof would only be ~10 hashes!


---

# Part 5: Proof-of-Work

## 5.1 What is Proof-of-Work?

Proof-of-Work is a consensus mechanism where miners compete to find a nonce (number used once) that, when included in the block header, produces a hash meeting a difficulty target.

**Target:** A hash must be numerically less than the target (more leading zeros = lower number)

**Difficulty:** Adjusted every 2,016 blocks to maintain ~10 minute block time

## 5.2 Simple Proof-of-Work Miner

In [27]:
def mine_block(data: str, difficulty: int) -> Tuple[int, str, float]:
    """
    Mine a block by finding a nonce that produces a hash with
    'difficulty' leading zeros.
    
    Args:
        data: Block data to hash
        difficulty: Number of leading zeros required
        
    Returns:
        Tuple of (nonce, hash, time_taken)
    """
    target = '0' * difficulty
    nonce = 0
    start_time = time.time()
    
    while True:
        block_data = f"{data}{nonce}"
        block_hash = sha256_hex(block_data)
        
        if block_hash.startswith(target):
            elapsed = time.time() - start_time
            return nonce, block_hash, elapsed
        
        nonce += 1

# Example: Mine a block with difficulty 4 (4 leading zeros)
block_data = "Block #12345: Alice -> Bob: 1 BTC"
difficulty = 4

print(f"Mining block with difficulty {difficulty}...\n")
nonce, block_hash, time_taken = mine_block(block_data, difficulty)

print(f"✓ Block mined!")
print(f"\nBlock data: {block_data}")
print(f"Nonce: {nonce:,}")
print(f"Hash: {block_hash}")
print(f"Time: {time_taken:.2f} seconds")
print(f"Hash rate: {nonce/time_taken:,.0f} hashes/second")

Mining block with difficulty 4...

✓ Block mined!

Block data: Block #12345: Alice -> Bob: 1 BTC
Nonce: 53,635
Hash: 000059e52859ec76cb9c8c037147d701dd58bf18f2f585c4ad2bbd460a08758d
Time: 0.04 seconds
Hash rate: 1,238,741 hashes/second


## 5.3 Difficulty Scaling

In [28]:
# Test different difficulty levels
difficulties = [3, 4, 5, 6]
block_data = "Test block"

print("Difficulty | Nonces Tried | Time (sec) | Hash Rate (H/s)")
print("-" * 65)

for diff in difficulties:
    nonce, hash_val, elapsed = mine_block(block_data, diff)
    hash_rate = nonce / elapsed if elapsed > 0 else 0
    print(f"{diff:10} | {nonce:12,} | {elapsed:10.2f} | {hash_rate:15,.0f}")

print(f"\n💡 Each additional zero roughly multiplies difficulty by 16 (16^n)")
print(f"   Bitcoin currently requires ~19 leading zeros!")

Difficulty | Nonces Tried | Time (sec) | Hash Rate (H/s)
-----------------------------------------------------------------
         3 |          481 |       0.00 |         326,767
         4 |       18,826 |       0.06 |         307,247
         5 |      469,133 |       0.27 |       1,724,994
         6 |   13,922,584 |       8.23 |       1,690,832

💡 Each additional zero roughly multiplies difficulty by 16 (16^n)
   Bitcoin currently requires ~19 leading zeros!


---

# Summary and Exercises

## What You've Learned

✓ **Hash Functions:** SHA-256, RIPEMD-160, and their properties  
✓ **Public Key Cryptography:** Key generation on secp256k1  
✓ **Digital Signatures:** ECDSA signing and verification  
✓ **Merkle Trees:** Building trees and generating proofs  
✓ **Proof-of-Work:** Mining and difficulty adjustment  
✓ **Bitcoin Addresses:** Complete address generation process  

## Exercises

Try these on your own to reinforce your understanding:

### Exercise 1: Vanity Address Generator

Create a function that generates Bitcoin addresses starting with a specific prefix (e.g., "1ABC").

**Hints:**
- Keep generating new key pairs
- Check if the address starts with your desired prefix
- Difficulty increases exponentially with prefix length

In [ ]:
def generate_vanity_address(prefix: str) -> Tuple[bytes, bytes, str]:
    """
    Generate a Bitcoin address starting with the given prefix.
    
    Args:
        prefix: Desired address prefix (e.g., "1ABC")
        
    Returns:
        Tuple of (private_key, public_key, address)
    """
    # YOUR CODE HERE
    pass

# Test with a short prefix
# vanity_prefix = "1A"  # Start small!
# priv, pub, addr = generate_vanity_address(vanity_prefix)
# print(f"Found address: {addr}")

### Exercise 2: Merkle Tree with More Transactions

Build a Merkle tree with 16 transactions and:
1. Calculate the proof size
2. Verify a proof for transaction #7
3. Try to verify with a tampered transaction

In [ ]:
# YOUR CODE HERE

# Create 16 transactions
transactions_large = [
    f"Transaction {i}: User{i} sends {i*0.1} BTC to User{i+1}"
    for i in range(16)
]

# Build tree, generate proof, verify
# ...

### Exercise 3: Multi-Signature Scheme (2-of-3)

Implement a simple 2-of-3 multi-signature scheme where any 2 of 3 parties must sign a message for it to be valid.

**Approach:**
1. Generate 3 key pairs
2. Have 2 of them sign the message
3. Verify that both signatures are valid

In [ ]:
# YOUR CODE HERE

# Generate 3 key pairs
# Sign message with 2 of them
# Verify both signatures
# Test: what if only 1 signature? What if signature is from wrong key?

### Exercise 4: Mining Difficulty Estimator

Write a function that estimates how long it would take to mine a block at a given difficulty, based on your computer's hash rate.

In [ ]:
def estimate_mining_time(difficulty: int, hash_rate: float) -> float:
    """
    Estimate time to mine a block.
    
    Args:
        difficulty: Number of leading zeros
        hash_rate: Hashes per second
        
    Returns:
        Estimated time in seconds
    """
    # YOUR CODE HERE
    pass

# Test your estimator
# Compare estimate vs. actual mining time

---

## Next Steps

Congratulations! You've mastered the cryptographic foundations of blockchain technology.

**Continue to:**
- **Section 2:** Bitcoin Deep Dive - Technical Architecture & Economics
- **Submodule 1B:** Bitcoin Blockchain Analysis - Analyzing real blockchain data

**Additional Resources:**
- [Bitcoin Developer Guide](https://bitcoin.org/en/developer-guide)
- [Mastering Bitcoin - Chapter 4: Keys and Addresses](https://github.com/bitcoinbook/bitcoinbook)
- [NIST FIPS 180-4: Secure Hash Standard](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)
- [SEC 2: Recommended Elliptic Curve Domain Parameters](https://www.secg.org/sec2-v2.pdf)